In [1]:
import jax
import jax.numpy as jnp
import functions
import pybamm
import numpy as np

In [3]:
# Set random keys
main_key = jax.random.PRNGKey(0)
key_train, key_test = jax.random.split(main_key)

params_bat = pybamm.ParameterValues("Chen2020")
# params_bat["Lower voltage cut-off [V]"] = -100.0
# params_bat["Upper voltage cut-off [V]"] = 100.0
# params_bat["Maximum concentration in negative electrode [mol.m-3]"] = 100e6
# params_bat["Maximum concentration in positive electrode [mol.m-3]"] = 100e6
# # params_bat["Minimum concentration in negative electrode [mol.m-3]"] = -100e6
# # params_bat["Minimum concentration in positive electrode [mol.m-3]"] = -100e6


# Hyperparameters
num_train = 100
num_test = 10

C = params_bat["Nominal cell capacity [A.h]"]

t_max = 3600
num_samples_I = 75
num_samples_c0 = 20

t = np.linspace(0, t_max, num_samples_I)
r = np.linspace(0, 1, num_samples_c0)

In [4]:
spm = pybamm.lithium_ion.SPM()
spm.events = []
Ran = params_bat["Negative particle radius [m]"]
Rca = params_bat["Positive particle radius [m]"]

def get_targets(I_samples, soc = 0.5, Dan=3.3e-14, Dca=4.0e-15):
    set_targets_anode = []
    set_targets_cathode = []
    set_c0_anode = []
    set_c0_cathode = []
    set_currents = []

    #I_pruned = []

    #fast_solver = pybamm.CasadiSolver(atol=1e-3, rtol=1e-3, mode="fast")
    
    for id, I_func in enumerate(I_samples):
        # Update parameter
        params_bat = pybamm.ParameterValues("Chen2020")
        #solver = pybamm.CasadiSolver(dt_max=1)
        params_bat["Current function [A]"] = pybamm.Interpolant(t, -1.* I_func, pybamm.t)            
        params_bat["Negative particle diffusivity [m2.s-1]"] = Dan
        params_bat["Positive particle diffusivity [m2.s-1]"] = Dca
        # Re-create the simulation with updated parameters
        sim = pybamm.Simulation(spm, parameter_values=params_bat)#, solver=fast_solver)
        sol = sim.solve(initial_soc=soc, t_eval=t)

        c0_anode = sol["Negative particle concentration"].entries[:,0,0]
        cn_target_anode = sol["Negative particle concentration"].entries[:,0,:]
            
        c0_cathode = sol["Positive particle concentration"].entries[:,0,0]
        cn_target_cathode = sol["Positive particle concentration"].entries[:,0,:]

            # if cn_target.shape[-1] != num_samples_I:
            #     continue
            # else:
            #     I_pruned.append(I_func)

            #I_pruned.append(I_func)
            
        set_targets_anode.append(cn_target_anode)
        set_c0_anode.append(c0_anode)

        set_targets_cathode.append(cn_target_cathode)
        set_c0_cathode.append(c0_cathode)
        set_currents.append(I_func)
    
    return set_targets_anode, set_c0_anode, set_targets_cathode, set_c0_cathode, set_currents


def generate_data(key, num, family = "CC"):
    keys = jax.random.split(key, num)
    train_I = []

    if family == "GRF":
        for k in keys:
            # Generate a random current function
            I_func = functions.GaussianRFCurrent(k, C, t_max)
            train_I.append(I_func(t))

    if family == "Triangle":
        for k in keys:
            # Sample a random amplitude value for the triangle current
            value = jax.random.uniform(k, shape=(), minval=-1, maxval=1)
            I_func = functions.TriangleCurrent(value * C)
            train_I.append(I_func(t))

    if family == "CC":
        for k in keys:
            # Generate a random constant current value between -1C and 1C
            value = jax.random.uniform(k, shape=(), minval=-1, maxval=1)
            I_func = functions.ConstantCurrent(value*C)
            # Evaluate at times t and append
            train_I.append(I_func(t))

    return train_I


In [5]:
from scipy.stats import qmc

# def get_diffusivity_sobol(num_samples, log_lower = -14, log_upper = -12):

#     # Reset the sampler (or create a new one) for a fresh Sobol sequence
#     sampler_log = qmc.Sobol(d=2, scramble=True)
#     samples_log = sampler_log.random_base2(m=int(np.log2(num_samples)))
#     # Scale the samples to the log-space range
#     log_samples = qmc.scale(samples_log, log_lower, log_upper)
#     # Transform back to the actual diffusivity values
#     #Ds_samples_log = 10 ** log_samples

#     return log_samples

import numpy as np
from scipy.stats import qmc

def get_soc_and_diffusivities_sobol(
    soc_levels,
    num_samples_per_soc,
    log_lower=-14,
    log_upper=-12
):
    """
    Generates diffusivities (D1, D2) in log10 space for each SOC level,
    ensuring each SOC gets a distinct chunk of the Sobol sequence.

    Returns an array of shape (len(soc_levels)*num_samples_per_soc, 3),
    with columns [SOC, D1, D2].
    """
    # We'll use a single 2D Sobol generator (for D1, D2 in log-space).
    sampler_log = qmc.Sobol(d=2, scramble=True)

    # Prepare output storage
    n_soc = len(soc_levels)
    total_samples = n_soc * num_samples_per_soc
    results = np.zeros((total_samples, 3), dtype=float)

    idx = 0  # row index in results
    for i, soc in enumerate(soc_levels):
        # Fast-forward in the Sobol sequence so each SOC gets its own block
        skip_from = (i+1) * num_samples_per_soc

        # Reset the sampler's internal state, then jump ahead
        sampler_log.reset()
        # print(i, skip_from)

        sampler_log.fast_forward(skip_from)

        # Generate the next chunk of samples in [0,1]^2
        # We assume num_samples_per_soc is a power of 2 for random_base2
        # e.g. 128, 256, 512, etc. Otherwise, you can use sampler_log.random(n).
        # samples_unit = sampler_log.random_base2(
        #     m=int(np.log2(num_samples_per_soc))
        # )
        samples_unit = sampler_log.random(num_samples_per_soc)


        # Scale from [0,1] -> [log_lower, log_upper] (for log10(D))
        samples_log = qmc.scale(samples_unit, log_lower, log_upper)

        # Convert from log10(D) to actual D
        #   D = 10^(log_samples)
        D_values = samples_log

        # Fill results with [SOC, D1, D2]
        for row in range(num_samples_per_soc):
            results[idx, 0] = soc
            results[idx, 1] = D_values[row, 0]  # D1
            results[idx, 2] = D_values[row, 1]  # D2
            idx += 1

    return results

In [6]:
socs = [0.,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.]
families = ["CC","GRF","Triangle"]
samples_per_soc = 256
socs, Dans, Dcas = get_soc_and_diffusivities_sobol(socs, samples_per_soc, log_lower = -15, log_upper = -12).T

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: log_samples (linear axes)
ax[0].scatter(Dans, Dcas, c='blue', alpha=0.6)
ax[0].set_xlabel('log10(D_s) Electrode 1')
ax[0].set_ylabel('log10(D_s) Electrode 2')
ax[0].set_title('Scatter Plot of log_samples (linear axes)')

# Plot 2: Ds_samples (log scale axes)
ax[1].scatter(10**Dans, 10**Dcas, c='green', alpha=0.6)
ax[1].set_xscale('log')
ax[1].set_yscale('log')
ax[1].set_xlabel('D_s Anode')
ax[1].set_ylabel('D_s Cathode')
ax[1].set_title('Scatter Plot of Ds_samples (log scale axes)')

plt.tight_layout()
plt.show()

In [ ]:
for family in families:

        # train_I = generate_data(key_train, num_train, family = family)
        # test_I = generate_data(key_test, num_test, family = family)

        list_train_current = []
        list_test_current = []

        list_train_cn_anode = []
        list_train_cn_cathode = []

        list_train_c0_anode = []
        list_train_c0_cathode = []

        list_test_cn_anode = []
        list_test_cn_cathode = []

        list_test_c0_anode = []
        list_test_c0_cathode = []
        
        for i, (soc, Dan_log, Dca_log) in enumerate(zip(socs, Dans, Dcas)):

                print(f"SOC: {soc}, Dan_log: {Dan_log}, Dca_log: {Dca_log}")
                train_key_i = jax.random.fold_in(key_train, i)
                test_key_i  = jax.random.fold_in(key_test,  i)
                Dan = 10**Dan_log
                Dca = 10**Dca_log

                train_I = generate_data(key_train, num_train, family = family)
                test_I = generate_data(key_test, num_test, family = family)

                try:
                        train_cn_anode, train_c0_anode, train_cn_cathode, train_c0_cathode, train_I2 = get_targets(train_I, soc = soc, Dan = Dan, Dca = Dca)
                        #Streng genommen stimmen hier Ds von test und training überein -> TODO Echter zufall
                        test_cn_anode, test_c0_anode, test_cn_cathode, test_c0_cathode, test_I2 = get_targets(test_I, soc = soc, Dan = Dan, Dca = Dca)
                except pybamm.SolverError as e:
                        print(f"Simulation for Dan,Dca = {Dan_log, Dca_log} failed. Skipping this run.")
                        continue

                list_train_current.extend(train_I2)
                #print(list_train_current)
                list_test_current.extend(test_I2)

                list_train_cn_anode.extend(train_cn_anode)
                list_train_cn_cathode.extend(train_cn_cathode)

                list_train_c0_anode.extend(train_c0_anode)
                list_train_c0_cathode.extend(train_c0_cathode)

                list_test_cn_anode.extend(test_cn_anode)
                list_test_cn_cathode.extend(test_cn_cathode)

                list_test_c0_anode.extend(test_c0_anode)
                list_test_c0_cathode.extend(test_c0_cathode)

        train_I = np.array(list_train_current)
        test_I = np.array(list_test_current)

        train_cn_anode = np.array(list_train_cn_anode)
        test_cn_anode = np.array(list_test_cn_anode)
        train_c0_anode = np.array(list_train_c0_anode)
        test_c0_anode = np.array(list_test_c0_anode)

        train_cn_cathode = np.array(list_train_cn_cathode)
        test_cn_cathode = np.array(list_test_cn_cathode)
        train_c0_cathode = np.array(list_train_c0_cathode)
        test_c0_cathode = np.array(list_test_c0_cathode)

        filename = family

        path = "data/spm/diff_D/"
        np.savez(path + "train/" + filename + str(num_train*samples_per_soc) + ".npz", current=train_I, initial_concentration_anode=train_c0_anode, target_concentration_anode=train_cn_anode,
                initial_concentration_cathode=train_c0_cathode, target_concentration_cathode=train_cn_cathode, diffusivity_anode = Dans, diffusivity_cathode = Dcas, soc = socs)
        np.savez(path + "test/" + filename + str(num_test*samples_per_soc) + ".npz", current=test_I, initial_concentration_anode=test_c0_anode, target_concentration_anode=test_cn_anode,
                initial_concentration_cathode=test_c0_cathode, target_concentration_cathode=test_cn_cathode, diffusivity_anode = Dans, diffusivity_cathode = Dcas, soc=socs)

In [ ]:
10**Dans

In [ ]:
train_data = np.load(path + "train/" + filename + str(num_train*samples_per_soc) + ".npz")
test_data = np.load(path + "test/" + filename + str(num_test*samples_per_soc) + ".npz")

In [ ]:
import matplotlib.pyplot as plt

# Suppose train_data["current"] is a numpy array of shape (14, 75),
# and we have some known battery capacity in ampere-hours, e.g. C = 5.0
data = train_data["current"][:5]

# Create a figure with one Axes, then a twin Axes for second scale
fig, ax1 = plt.subplots(figsize=(8, 5))

# Plot the currents on ax1
for i in range(data.shape[0]):
    ax1.plot(data[i, :], label=f"Series {i}")

ax1.set_xlabel("Index (time steps)")
ax1.set_ylabel("Current [A]", color="blue")
ax1.tick_params(axis='y', labelcolor="blue")

# Create second y‐axis for the C‐rate
ax2 = ax1.twinx()
for i in range(data.shape[0]):
    # Convert current [A] into C‐rate by dividing by capacity [Ah]
    # (If your capacity is in different units, adjust accordingly)
    c_rate = data[i, :] / C
    ax2.plot(c_rate, linestyle="--", alpha=0.7)

ax2.set_ylabel("C-rate", color="red")
ax2.tick_params(axis='y', labelcolor="red")

plt.title("Currents (left axis) and C-rates (right axis)")
plt.show()


In [ ]:
train_data["current"].shape

In [ ]:
data = train_data["target_concentration_anode"]

plt.figure(figsize=(8, 5))

for i in range(data.shape[0]):  # i goes from 0 to 13
    plt.plot(data[i,-1, :], label=f"Series {i}")

plt.xlabel("Index (time steps)")
plt.ylabel("Concentration Value Normalised")
#plt.title("All 14 Current Traces in One Plot")
#plt.legend()
plt.show()

In [ ]:
data = train_data["target_concentration_cathode"]

plt.figure(figsize=(8, 5))

for i in range(data.shape[0]):  # i goes from 0 to 13
    plt.plot(data[i,-1, :], label=f"Series {i}")

plt.xlabel("Index (time steps)")
plt.ylabel("Concentration Value Normalised")
#plt.title("All 14 Current Traces in One Plot")
#plt.legend()
plt.show()

In [ ]:
train_data["target_concentration_cathode"].shape